# Baby Name Visualisations in France (1900–2020)

This notebook implements the **3 interactive visualisations** for the Week 2 project, based on the INSEE baby names dataset by department.

| # | Theme | Visualisation type |
|---|---|---|
| 1 | Temporal evolution | Interactive line chart (click on legend) |
| 2 | Regional effect | Choropleth map with name selector |
| 3 | Gender effects | M/F proportion chart with name selector |

In [10]:
import altair as alt
import pandas as pd
import geopandas as gpd

# Allow Altair to handle large datasets (>5000 rows)
alt.data_transformers.enable('json')

DataTransformerRegistry.enable('json')

## Data Loading and Cleaning

We remove:
- Aggregated rare names (`_PRENOMS_RARES`)
- Unknown departments (`XX`)
- Unknown years (`XXXX`)

In [11]:
# --- Load baby names ---
names = pd.read_csv("dpt2020.csv", sep=";", dtype=str)

# Cleaning
names = names[names['preusuel'] != '_PRENOMS_RARES']
names = names[names['dpt'] != 'XX']
names = names[names['annais'] != 'XXXX']

names['annais'] = names['annais'].astype(int)
names['nombre'] = pd.to_numeric(names['nombre'], errors='coerce')
names['sexe']   = names['sexe'].astype(int)
names.dropna(subset=['nombre'], inplace=True)
names['nombre'] = names['nombre'].astype(int)

# Rename columns for readability
names = names.rename(columns={'preusuel': 'Prénom', 'annais': 'Années'})

# --- Load geographic boundaries ---
depts = gpd.read_file('departements-version-simplifiee.geojson')

print(f"Names loaded : {len(names):,} rows")
print(f"Departments  : {len(depts)}")
names.sample(3)

Names loaded : 3,668,274 rows
Departments  : 96


,sexe,Prénom,Années,dpt,nombre
1648531,1,WAËL,2016,59,6
3508908,2,SAMIRA,1995,84,6
3039849,2,MARIA,1952,59,46


---
## Visualisation 1 — Temporal Evolution of Baby Names

**Questions:** How do baby names evolve over time? Are some names consistently popular? Did any names experience a sudden, brief peak in popularity? Are there trends over time?

**Design choice:** Two-layer interactive line chart with two independent interactions:

- **Coloured top 20 lines** — the 20 most popular names, each with its own colour, permanently visible.  
  → **Click a name in the legend** to focus on it: all other lines grey out, making it easy to isolate a trajectory.

- **Red search line** — a dropdown lets you search any name from a pool of 500+ (including names with a sharp yearly peak ≥ 300 births, capturing briefly popular names).  
  → The selected name is overlaid in bold red on top of the top 20, allowing direct comparison.

In [12]:
# National aggregation by (name, year)
names_time = names.groupby(['Prénom', 'Années'], as_index=False)['nombre'].sum()

# --- Background layer: top 20 names by cumulative total ---
top20 = (
    names_time.groupby('Prénom')['nombre']
    .sum()
    .nlargest(20)
    .index.tolist()
)
viz1_background = names_time[names_time['Prénom'].isin(top20)].copy()

# --- Search pool: top 500 cumulative + names with a notable yearly peak ---
# A name with a sharp but brief peak may rank low cumulatively but high at its peak year
name_totals = names_time.groupby('Prénom')['nombre'].sum()
name_peaks  = names_time.groupby('Prénom')['nombre'].max()

search_pool = sorted(
    set(name_totals.nlargest(500).index) |
    set(name_peaks[name_peaks >= 300].index)
)
viz1_search = names_time[names_time['Prénom'].isin(search_pool)].copy()

print(f"Background : {len(top20)} names (top 20 cumulative)")
print(f"Search pool: {len(search_pool)} names (top 500 + peak ≥ 300/year)")

Background : 20 names (top 20 cumulative)
Search pool: 1145 names (top 500 + peak ≥ 300/year)


In [13]:
x_ticks = list(range(1900, 2021, 10))

# Pan (click+drag) and zoom (scroll) on both axes
zoom = alt.selection_interval(bind='scales', encodings=['x', 'y'])

# --- Layer 1: top 20 coloured lines + legend click to grey others ---
# Click a name in the legend to isolate it; double-click to reset all.
legend_sel = alt.selection_point(fields=['Prénom'], bind='legend')

top20_layer = (
    alt.Chart(viz1_background)
    .mark_line(point=alt.OverlayMarkDef(size=40))
    .encode(
        x=alt.X(
            'Années:O',
            title='Year',
            axis=alt.Axis(values=x_ticks, labelAngle=-45),
        ),
        y=alt.Y('nombre:Q', title='Number of births'),
        color=alt.condition(
            legend_sel,
            alt.Color('Prénom:N', title='Name', legend=alt.Legend(title='Click to filter')),
            alt.value('lightgrey'),
        ),
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.15)),
        strokeWidth=alt.condition(legend_sel, alt.value(2.5), alt.value(0.8)),
        tooltip=[
            alt.Tooltip('Prénom:N', title='Name'),
            alt.Tooltip('Années:O',   title='Year'),
            alt.Tooltip('nombre:Q',   title='Births', format=','),
        ],
    )
    .add_params(legend_sel, zoom)
)

# --- Layer 2: dropdown to search any name (shown in bold red) ---
dropdown1 = alt.binding_select(options=search_pool, name='Search any name: ')
name_sel1 = alt.selection_point(
    fields=['Prénom'],
    bind=dropdown1,
    value=search_pool[0],
)

search_layer = (
    alt.Chart(viz1_search)
    .mark_line(strokeWidth=3.5, point=alt.OverlayMarkDef(size=70, color='#E45756'))
    .encode(
        x=alt.X(
            'Années:O',
            title='Year',
            axis=alt.Axis(values=x_ticks, labelAngle=-45),
        ),
        y=alt.Y('nombre:Q', title='Number of births'),
        color=alt.value('#E45756'),
        tooltip=[
            alt.Tooltip('Prénom:N', title='Name'),
            alt.Tooltip('Années:O',   title='Year'),
            alt.Tooltip('nombre:Q',   title='Births', format=','),
        ],
    )
    .add_params(name_sel1)
    .transform_filter(name_sel1)
)

chart_viz1 = (top20_layer + search_layer).properties(
    width=860,
    height=420,
    title=alt.TitleParams(
        'Baby name evolution — top 20 coloured (click legend to grey others) + any name in red',
        fontSize=14,
    ),
)

chart_viz1

alt.LayerChart(...)

---
## Visualisation 2 — Regional Effect (Choropleth Map)

**Questions:** Are some names more popular in certain regions? Are popular names uniformly popular across the whole country?

**Design choice:** Choropleth map with two independent controls:
- **Dropdown** — select any of the top 200 names.
- **Year slider** — move from 1900 to 2020 to see how the regional distribution of a name shifts over time.

→ The colour encodes births **per 1,000 births** in the department for the selected year (normalised to remove population-size bias).  
→ Departments with no recorded births for that name in that year appear as absent (informative: the name was not used there that year).

In [14]:
# Aggregation by (department, name, year)
grouped_geo_year = names.groupby(['dpt', 'Prénom', 'Années'], as_index=False)['nombre'].sum()

# Total births per department per year for normalisation
dept_totals_year = (
    names.groupby(['dpt', 'Années'], as_index=False)['nombre']
    .sum()
    .rename(columns={'nombre': 'total_dept'})
)
grouped_geo_year = grouped_geo_year.merge(dept_totals_year, on=['dpt', 'Années'])
grouped_geo_year['pour_mille'] = (
    grouped_geo_year['nombre'] / grouped_geo_year['total_dept'] * 1000
).round(3)

# Top 200 names for the dropdown
top200 = (
    grouped_geo_year.groupby('Prénom')['nombre']
    .sum()
    .nlargest(200)
    .index.sort_values()
    .tolist()
)

# Attribute lookup table — NO geometry, just the compound key + values.
# Key pattern: "{dpt}_{Prénom}_{Années}"
# Used at render time via transform_calculate + transform_lookup (see chart cell).
viz2_lookup = grouped_geo_year[grouped_geo_year['Prénom'].isin(top200)].copy()
viz2_lookup['key'] = (
    viz2_lookup['dpt'] + '_' +
    viz2_lookup['Prénom'] + '_' +
    viz2_lookup['Années'].astype(str)
)
viz2_lookup = viz2_lookup[['key', 'nombre', 'pour_mille']]

# The 96-row GeoDataFrame `depts` (loaded in cell 4) is used directly as the
# chart data source — Altair serialises it as a GeoJSON FeatureCollection,
# which mark_geoshape can render correctly.

# Calculate centroids for text annotations
depts_with_centroids = depts.copy()
depts_with_centroids['longitude'] = depts_with_centroids.geometry.centroid.x
depts_with_centroids['latitude'] = depts_with_centroids.geometry.centroid.y

print(f"Viz 2 lookup: {len(viz2_lookup):,} rows | {len(top200)} names")
print(f"Dept source : {len(depts)} department polygons")

Viz 2 lookup: 1,151,368 rows | 200 names
Dept source : 96 department polygons


/tmp/ipykernel_2487/4133405244.py:41: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  depts_with_centroids['longitude'] = depts_with_centroids.geometry.centroid.x
/tmp/ipykernel_2487/4133405244.py:42: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  depts_with_centroids['latitude'] = depts_with_centroids.geometry.centroid.y


In [15]:
# Use alt.param (not selection_point) for the name so the selected value is
# exposed as a plain Vega signal called 'selected_name', which can be used
# directly in transform_calculate expressions.
name_param = alt.param(
    name='selected_name',
    value=top200[0],
    bind=alt.binding_select(options=top200, name='Name: '),
    # bind=alt.binding(input='search', name='Search name: ', placeholder='Type a name...'),
)

year_param = alt.param(
    name='year_val',
    value=1970,
    bind=alt.binding_range(min=1900, max=2020, step=1, name='Year: '),
)

# Base geoshape layer
base_map = (
    # Base data: 96-row GeoDataFrame → Altair serialises as GeoJSON FeatureCollection
    # → mark_geoshape renders each feature correctly.
    alt.Chart(depts_with_centroids)
    .mark_geoshape(stroke='white', strokeWidth=0.5)
    .encode(
        # Departments with no data (unmatched lookup → pour_mille is null) → grey
        color=alt.condition(
            'isValid(datum.pour_mille)',
            alt.Color(
                'pour_mille:Q',
                scale=alt.Scale(scheme='blues'),
                title='Births per 1,000',
                legend=alt.Legend(gradientLength=220),
            ),
            alt.value('lightgrey'),
        ),
        tooltip=[
            alt.Tooltip('nom:N',        title='Department'),
            alt.Tooltip('nombre:Q',     title='Births',          format=','),
            alt.Tooltip('pour_mille:Q', title='Per 1,000 births', format='.2f'),
        ],
    )
    # Step 1: build a compound key per department using the current param values.
    # Vega expression has direct access to signals 'selected_name' and 'year_val'.
    .transform_calculate(
        lookup_key="datum.code + '_' + selected_name + '_' + toString(year_val)",
    )
    # Step 2: attach 'nombre' and 'pour_mille' by matching the compound key in viz2_lookup.
    # Unmatched departments get null values → rendered grey by the condition above.
    .transform_lookup(
        lookup='lookup_key',
        from_=alt.LookupData(viz2_lookup, 'key', ['nombre', 'pour_mille']),
    )
    .add_params(name_param, year_param)
    .project(type='mercator')
)

# Text annotations showing the normalized rate (per 1,000) in each department
text_layer = (
    alt.Chart(depts_with_centroids)
    .mark_text(fontSize=9, fontWeight='bold', color='#222222', dy=-2)
    .encode(
        longitude='longitude:Q',
        latitude='latitude:Q',
        text=alt.condition(
            'isValid(datum.pour_mille) && datum.pour_mille > 0',
            alt.Text('pour_mille:Q', format='.1f'),
            alt.value(''),
        ),
        opacity=alt.condition(
            'isValid(datum.pour_mille) && datum.pour_mille > 0',
            alt.value(0.75),
            alt.value(0),
        ),
    )
    .transform_calculate(
        lookup_key="datum.code + '_' + selected_name + '_' + toString(year_val)",
    )
    .transform_lookup(
        lookup='lookup_key',
        from_=alt.LookupData(viz2_lookup, 'key', ['nombre', 'pour_mille']),
    )
    .project(type='mercator')
)

chart_viz2 = (base_map + text_layer).properties(
    width=900,
    height=750,
    title=alt.TitleParams(
        'Regional distribution of a name by year (births per 1,000 in the department)',
        fontSize=14,
    ),
)

chart_viz2

alt.LayerChart(...)

---
## Visualisation 3 — Gender Effects (Proportion of Gender-Neutral Names)

**Questions:** Are there gender effects in the data? Does the share of gender-neutral names evolve over time? Do individual neutral names follow the same trend?

**Design choice:** Line chart showing the proportion of births given to gender-neutral names (those given to both sexes at more than 5%) over time.  
→ Dropdown to show either the **aggregate** trend across all unisex names (`All`) or zoom in on a **single name** from the list.  
→ The proportion axis makes it easy to compare years despite varying total birth counts.

In [16]:
# National aggregation by (name, year, sex)
gender_time = names.groupby(['Prénom', 'Années', 'sexe'], as_index=False)['nombre'].sum()

# Pivot: one column per sex
pivot = (
    gender_time
    .pivot_table(index=['Prénom', 'Années'], columns='sexe', values='nombre', fill_value=0)
    .reset_index()
)
pivot.columns.name = None
pivot = pivot.rename(columns={1: 'male', 2: 'female'})
pivot['total'] = pivot['male'] + pivot['female']

# Identify unisex names (5–95% for either sex, total > 1,000)
mixed_stats = pivot.groupby('Prénom').agg(
    masc=('male', 'sum'),
    fem=('female', 'sum'),
).reset_index()
mixed_stats['total'] = mixed_stats['masc'] + mixed_stats['fem']
mixed_stats['prop_f'] = mixed_stats['fem'] / mixed_stats['total']

mixed_names = (
    mixed_stats[
        (mixed_stats['prop_f'] > 0.05) &
        (mixed_stats['prop_f'] < 0.95) &
        (mixed_stats['total'] > 1000)
    ]
    .nlargest(200, 'total')   # top 200 unisex names by total births
    ['Prénom']
    .sort_values()
    .tolist()
)

print(f"Unisex names identified: {len(mixed_names)}")

Unisex names identified: 55


In [17]:
# Prepare data: aggregate by year for "All" view
names['is_neutral'] = names['Prénom'].isin(mixed_names)
neutral_evolution = names.groupby('Années').agg(
    neutral_births=('nombre', lambda x: x[names.loc[x.index, 'is_neutral']].sum()),
    total_births=('nombre', 'sum')
).reset_index()
neutral_evolution['proportion'] = neutral_evolution['neutral_births'] / neutral_evolution['total_births']
neutral_evolution['Prénom'] = 'All'

# Prepare data: individual neutral names by year with ALL years (1900-2020)
all_years = pd.DataFrame({'Années': range(1900, 2021)})
total_by_year = names.groupby('Années', as_index=False)['nombre'].sum().rename(columns={'nombre': 'total_births'})

# For each neutral name, ensure all years are present
neutral_by_name_list = []
for name in mixed_names:
    name_data = names[names['Prénom'] == name].groupby('Années', as_index=False)['nombre'].sum()
    # Merge with all years to fill missing ones
    complete_data = all_years.merge(name_data, on='Années', how='left')
    complete_data['nombre'] = complete_data['nombre'].fillna(0)
    complete_data['Prénom'] = name
    neutral_by_name_list.append(complete_data)

neutral_by_name = pd.concat(neutral_by_name_list, ignore_index=True)
neutral_by_name = neutral_by_name.merge(total_by_year, on='Années')
neutral_by_name = neutral_by_name.rename(columns={'nombre': 'neutral_births'})
neutral_by_name['proportion'] = neutral_by_name['neutral_births'] / neutral_by_name['total_births']

# Combine both datasets
neutral_data = pd.concat([neutral_evolution, neutral_by_name], ignore_index=True)

# Dropdown with "All" first, then sorted names
name_options = ['All'] + sorted(mixed_names)
name_dropdown = alt.binding_select(options=name_options, name='Filter by name: ')
name_filter = alt.selection_point(fields=['Prénom'], bind=name_dropdown, value='All')

# Visualization
chart_neutral = (
    alt.Chart(neutral_data)
    .mark_line(point=True, strokeWidth=2.5, color='#E45756')
    .encode(
        x=alt.X('Années:O', title='Year', axis=alt.Axis(values=list(range(1900, 2021, 10)), labelAngle=-45)),
        y=alt.Y('proportion:Q', title='Proportion of Gender-Neutral Names', axis=alt.Axis(format='%')),
        tooltip=[
            alt.Tooltip('Années:O', title='Year'),
            alt.Tooltip('neutral_births:Q', title='Gender-neutral births', format=','),
            alt.Tooltip('total_births:Q', title='Total births', format=','),
            alt.Tooltip('proportion:Q', title='Proportion', format='.4%')
        ]
    )
    .add_params(name_filter)
    .transform_filter(name_filter)
    .properties(
        width=860, 
        height=420, 
        title='Evolution of gender-neutral names over time (1900–2020)'
    )
)

chart_neutral

alt.Chart(...)